# Neural Networks and Deep Learning, MDS HSE

## Homework 4. Transformers.

### General Information

### Grading and Penalties

The maximum possible grade for the assignment without bonuses is 10 points. Submitting the work after the hard deadline is not allowed.

Submitting after the soft deadline incurs a penalty of -1 point per day. Twice per semester (two modules), students are allowed to use an extension and submit by the hard deadline without penalty.

The assignment must be completed individually. “Similar” solutions will be considered plagiarism, and all involved students (including those whose work was copied) will receive no more than 0 points for the assignment. If you find a solution (or part of it) to any task from an open source, you must include a link to that source in a separate section at the end of your work (most likely you won’t be the only one who found it, so providing the link helps avoid suspicion of plagiarism).

Inefficient code implementation may negatively affect your grade. The grade may also be reduced for poorly readable code and poorly formatted plots. All answers must be accompanied by either code or comments explaining how they were obtained.

Use of generative models is allowed under the following conditions:
- The amount of code generated by such models does not exceed 30% of the total.
- You specify the model used and the prompt.
- At the end of your work, you include a **reflection on your experience using generative AI for this homework:  
  Describe how often you had to fix the code yourself or ask the model to correct something. Was it faster than writing the code on your own?

If these requirements are not met, the assignment will not be graded, and the maximum possible score is 0 points.

### About the Assignment

In this homework assignment, you will add a decoder part to BERT and solve the task of writing tl;dr for news texts in Russian.

In addition to this, to get an excellent grade, you will need to implement a less greedy strategy for selecting the next token for generation.

In [ ]:
!pip install transformers datasets evaluate

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, BertModel, BertTokenizer

## Data preparation (0.5 points)

We will use the dataset from 🤗 Ilya Gusev's “gazeta.” It consists of pairs (full text of the news article -- its summary). 

You can read more about the dataset [here](https://huggingface.co/datasets/IlyaGusev/gazeta).



In [ ]:
# Let's load the data using the datasets library.
# You are free to take less or more data, but something around >=10% usually works best.

from datasets import load_dataset

dataset = load_dataset("IlyaGusev/gazeta", revision="v2.0", split="train[:10%]")

You зкщифидн remember that texts must be **tokenized** before being fed into the model.

Add padding to `max_length=512` for training data, and to `max_length=128` for labels.

Use truncation for texts whose length in tokens exceeds `max_length`.

In [ ]:
# Prepare data for the Bert model

model_name = "deepvk/bert-base-uncased"  # Which BERT to use

tokenizer = AutoTokenizer.from_pretrained(model_name)


def preprocess(examples, use_padding=True):

    # <YOUR CODE HERE>

    return model_inputs

In [ ]:
tokenized_dataset = dataset.map(preprocess, batched=False)
tokenized_dataset.set_format("torch")

Map:   0%|          | 0/3048 [00:00<?, ? examples/s]

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = #<YOUR CODE HERE>
eval_dataloader = #<YOUR CODE HERE>

## Implementing a Decoder Network (3 points)

In this section, you need to **implement your own decoder for generating text**.

You can draw inspiration from the code from the seminar. When initializing weights, it is worth (but not necessary) to remember the nuances.

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer

# BERT-based summarization model class with custom decoder


class BertSummarizer(nn.Module):
    def __init__(
        self,
        bert_model_name="bert-base-uncased",
        hidden_size=768,
        num_decoder_layers=3,
        num_heads=8,
        dropout=0.1,
    ):
        super().__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.hidden_size = hidden_size

        # Embeddings for tokens at the decoder input
        self.embedding = nn.Embedding(self.bert.config.vocab_size, hidden_size)

        # <YOUR CODE HERE>

    # Function for creating a mask to prevent peeking ahead in the decoder

    def generate_square_subsequent_mask(self, T):
        # <YOUR CODE HERE>
        pass

    def forward(self, input_ids, attention_mask, decoder_input_ids):
        encoder_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        memory = (
            encoder_outputs.last_hidden_state
        )  # BERT outputs for use in the decoder

        # Embeddings for decoder input tokens
        embedded = self.embedding(decoder_input_ids)

        # <YOUR CODE HERE>
        output = None  # change this line

        return self.softmax(output)

    def generate(self, input_ids, attention_mask, tokenizer, max_len=50):
        encoder_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        memory = encoder_outputs.last_hidden_state
        batch_size = input_ids.size(0)

        # Start with the token [CLS] or [BOS] (start of sequence)
        decoder_input_ids = torch.full(
            (batch_size, 1), tokenizer.cls_token_id, dtype=torch.long
        ).to(input_ids.device)
        memory = memory.transpose(0, 1)
        generated_tokens = []

        for _ in range(max_len):
            embedded = self.embedding(decoder_input_ids).transpose(0, 1)

            # Generatу a mask to prevent looking ahead
            decoder_attention_mask = self.generate_square_subsequent_mask(
                embedded.size(0)
            ).to(input_ids.device)
            decoder_output = self.decoder(
                tgt=embedded, memory=memory, tgt_mask=decoder_attention_mask
            )

            output = self.fc_out(decoder_output.transpose(0, 1))

            # Get the token index with the highest probability.
            # Remember, if EOS is predicted, we stop generating.

            # <YOUR CODE HERE>

        generated_sequence = tokenizer.decode(
            decoder_input_ids.squeeze().tolist(), skip_special_tokens=True
        )

        return generated_sequence

In [ ]:
# Let's initialize our model and examine its architecture.


model = BertSummarizer(bert_model_name=model_name)
model = model.to("cuda")
model

In [ ]:
# Let's look at generation without training

eval_data_sample = next(iter(eval_dataloader))
model.generate(
    eval_data_sample["input_ids"][:1].to("cuda"),
    eval_data_sample["attention_mask"][:1].to("cuda"),
    tokenizer,
)

'угле ##стеров ##ряз ##drop мнения прямои ##ru распах связа опове ##рогом необходимость подслу уходила коробка смарт шпион ##оцени реитинг учебник подош проидет 📝 правом англииском известные body подумали регла швеицар ##% ##ить шпион ##оцени реитинг помимо club12 ##ольше ##гант очертания ##лли ##зали собравшихся пошу группо мощныи хостинг удивлением настоящии υ'

## Model training (1 point)

0.25 points for the simplest working cycle; 

0.5 points for graphs for loss and metrics on the train and validation sets.

0.25 points for logging in TensorBoard or WandB

In this section, you need to **implement a cycle for model training**.

In [ ]:
# Example of training on a single iteration
# Everyone remembers that we need to predict the next token, right?


def train_step(
    model, input_ids, attention_mask, decoder_input_ids, optimizer, criterion
):
    model.train()
    optimizer.zero_grad()
    outputs = model(input_ids, attention_mask, decoder_input_ids)
    loss = criterion(outputs.view(-1, outputs.size(-1)), decoder_input_ids.view(-1))
    loss.backward()
    optimizer.step()

    return loss.item()

## Quality metrics (1 point)

0.33 points for each of the proposed metrics

**Implement a function to calculate the quality metrics of summarization.**

What we want to calculate:
 1. [HuggingFace Rouge](https://huggingface.co/spaces/evaluate-metric/rouge)
 2. [HuggingFace Bleu](https://huggingface.co/spaces/evaluate-metric/bleu)
 3. [HuggingFace BERT Score](https://huggingface.co/spaces/evaluate-metric/bertscore)

In [ ]:
def compute_metrics():
    # <YOUR CODE HERE>
    pass


def evaluation():
    # <YOUR CODE HERE>
    pass

## Model training (0.5 points)
**Train the model, save the best version** (using the `.save_pretrained()` method of the AutoModel... class or `torch.save()`) **and add a generation example**. Note that if the tokenizer has been changed (and it is better to just use the default), it must also be saved.

To compare the generation quality assessment based on the values of the implemented metrics, you can run ruT5-small. We intentionally provide the baseline in this form.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("YOUR MODEL")
summary = #<YOUR CODE HERE>

## Implementing less greedy strategies for selecting the next token (4 points)
Is selecting the most probable token at each step always the best strategy for text generation?

<details>
    <summary>Spoiler</summary>
    <p>No</p>
</details>

**Comparison of text generation strategies:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br><br/> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br><br/> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words <br/> that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br><br/> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps <br/> the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br><br/> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br><br/> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities <br/> by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br><br/> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br><br/> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br><br/> **Cons:** Computationally more complex and requires a good loss function. |

Docs:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

**1. Implement the Top-k strategy in the `generate` method** (1 point).   

**2. Implement the Nucleus Sampling (Top-p) strategy in the `generate` method** (1 point)

**3. Implement the Beam Search strategy** (2 points)

Was it possible to improve the generation?

## Bonus (1 point)

What you need to do:

- Take only the decoder part from the existing model
- Write a training cycle (or rather, improve the existing one) and tune the decoder
- Check the quality of generation using metrics and your own eyes
- Answer the question “Does the use of Encoder-Decoder architecture provide a significant boost in generation quality?” with proof